# Density Split Statistics Examples

This notebook demonstrates how to use the Density Split Statistics estimator from the ACM package.

Density split statistics divide the galaxy sample into quantiles based on the local density field and measure clustering separately for each quantile, providing information about non-Gaussian features.

In [ ]:
from pathlib import Path
import glob
import fitsio
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from acm.estimators.galaxy_clustering.density_split import DensitySplit
from acm import setup_logging

def get_hod_fns(cosmo=1, phase=0, redshift=0.5, seed=0):
    """
    Get the list of HOD file names for a given cosmology,
    phase, and redshift.
    """
    base_dir = '/pscratch/sd/n/ntbfin/emulator/hods/z0.5/yuan23_prior/'
    hod_dir = Path(base_dir) / f'c{cosmo:03}_ph{phase:03}/seed{seed}/'
    hod_fns = glob.glob(str(Path(hod_dir) / f'hod*.fits'))
    return sorted(hod_fns)

def get_hod_positions(filename, los='z'):
    """Get redshift-space positions from a HOD file."""
    hod, header = fitsio.read(filename, header=True)
    qpar, qperp = header['Q_PAR'], header['Q_PERP']
    if los == 'x':
        pos = np.c_[hod['X_RSD'], hod['Y_PERP'], hod['Z_PERP']]
        boxsize = np.array([2000/qpar, 2000/qperp, 2000/qperp])
    elif los == 'y':
        pos = np.c_[hod['X_PERP'], hod['Y_RSD'], hod['Z_PERP']]
        boxsize = np.array([2000/qperp, 2000/qpar, 2000/qperp])
    elif los == 'z':
        pos = np.c_[hod['X_PERP'], hod['Y_PERP'], hod['Z_RSD']]
        boxsize = np.array([2000/qperp, 2000/qperp, 2000/qpar])
    return pos, boxsize

def get_box_args(boxsize, cellsize):
    meshsize = (boxsize / cellsize).astype(int)
    return dict(boxsize=boxsize, boxcenter=0.0, meshsize=meshsize)


setup_logging()

# load galaxy catalog for testing
hod_fn = get_hod_fns(cosmo=0, phase=0, redshift=0.5)[0]
positions, boxsize = get_hod_positions(hod_fn, los='z')

## Setting up the Density Split Estimator

We first create the density field and split it into quantiles based on local density values.

In [ ]:
# Set up box parameters with a finer mesh for density field
box_args = get_box_args(boxsize, cellsize=10)

# Instantiate the DensitySplit class
ds = DensitySplit(data_positions=positions, backend='jaxpower', **box_args)

# Set the density contrast field with a smoothing radius
smoothing_radius = 10  # Mpc/h
ds.set_density_contrast(smoothing_radius=smoothing_radius)

# Split into quantiles
nquantiles = 5
ds.set_quantiles(nquantiles=nquantiles, query_method='randoms')

print(f"Divided sample into {nquantiles} density quantiles")
print(f"Smoothing radius: {smoothing_radius} Mpc/h")

## Visualizing the Quantiles

Let's visualize the distribution of densities in each quantile.

In [ ]:
# Plot the density distribution for each quantile
# (you can also save the plot to disk by providing
# a filename to the function)
ds.plot_quantiles()
plt.show()

## Auto-Correlation Functions by Quantile

Compute the two-point correlation function for each density quantile.

In [ ]:
# Define separation bins
sedges = np.arange(0, 50, 1)  # separation in Mpc/h
muedges = np.linspace(-1, 1, 241)  # angular bins
edges = (sedges, muedges)

# Compute auto-correlation for each quantile
acf = ds.quantile_correlation(edges=edges, los='z', nthreads=16, gpu=False)

# Plot the correlation function multipoles
ds.plot_quantile_correlation()
plt.show()

## Extracting Multipoles

We can extract specific multipoles (monopole, quadrupole) for each quantile.

In [ ]:
# Extract multipoles for each quantile
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i in range(nquantiles):
    s, multipoles = acf[i](ells=[0, 2], return_sep=True)
    
    # Plot monopole
    axes[0].plot(s, s**2 * multipoles[0], label=f'Q{i+1}', alpha=0.8)
    
    # Plot quadrupole
    axes[1].plot(s, s**2 * multipoles[1], label=f'Q{i+1}', alpha=0.8)

axes[0].set_xlabel('s [Mpc/h]')
axes[0].set_ylabel(r'$s^2 \xi_0(s)$ [Mpc/h]$^2$')
axes[0].set_title('Monopole')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('s [Mpc/h]')
axes[1].set_ylabel(r'$s^2 \xi_2(s)$ [Mpc/h]$^2$')
axes[1].set_title('Quadrupole')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Cross-Correlation with Full Sample

Compute the cross-correlation between each quantile and the full galaxy sample.

In [ ]:
# Compute quantile-data cross-correlation
ccf = ds.quantile_data_correlation(positions, edges=edges, los='z', nthreads=16, gpu=False)

# Plot the cross-correlation function
ds.plot_quantile_data_correlation()
plt.show()

## Cross-Correlation Multipoles

Extract and plot the cross-correlation multipoles.

In [ ]:
# Extract cross-correlation multipoles
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i in range(nquantiles):
    s, multipoles = ccf[i](ells=[0, 2], return_sep=True)
    
    # Plot monopole
    axes[0].plot(s, s**2 * multipoles[0], label=f'Q{i+1}', alpha=0.8)
    
    # Plot quadrupole
    axes[1].plot(s, s**2 * multipoles[1], label=f'Q{i+1}', alpha=0.8)

axes[0].set_xlabel('s [Mpc/h]')
axes[0].set_ylabel(r'$s^2 \xi_0^{\rm cross}(s)$ [Mpc/h]$^2$')
axes[0].set_title('Cross-Correlation Monopole')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('s [Mpc/h]')
axes[1].set_ylabel(r'$s^2 \xi_2^{\rm cross}(s)$ [Mpc/h]$^2$')
axes[1].set_title('Cross-Correlation Quadrupole')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Comparing Different Backends

The DensitySplit estimator supports both `jaxpower` and `pypower` backends.

In [ ]:
# Compute with both backends
backends = ['jaxpower', 'pypower']
acf_dict = {}

for backend in backends:
    ds_temp = DensitySplit(data_positions=positions, backend=backend, **box_args)
    ds_temp.set_density_contrast(smoothing_radius=smoothing_radius)
    ds_temp.set_quantiles(nquantiles=nquantiles, query_method='randoms')
    acf_dict[backend] = ds_temp.quantile_correlation(edges=edges, los='z', nthreads=4, gpu=True)

# Compare monopole for the lowest density quantile (Q1)
fig, ax = plt.subplots(figsize=(8, 4))

for backend, acf_result in acf_dict.items():
    s, multipoles = acf_result[0](ells=[0], return_sep=True)
    ls = '-' if backend == 'jaxpower' else '--'
    ax.plot(s, s**2 * multipoles[0], label=backend, ls=ls, linewidth=2, alpha=0.8)

ax.set_xlabel('s [Mpc/h]')
ax.set_ylabel(r'$s^2 \xi_0(s)$ [Mpc/h]$^2$')
ax.set_title('Backend Comparison (Q1 Monopole)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Effect of Smoothing Scale

The smoothing radius affects how the density field is defined. Let's compare different smoothing scales.

In [ ]:
# Test different smoothing radii
smoothing_radii = [5, 10, 20]  # Mpc/h
colors = ['blue', 'green', 'red']

fig, ax = plt.subplots(figsize=(8, 5))

for R, color in zip(smoothing_radii, colors):
    ds_temp = DensitySplit(data_positions=positions, backend='jaxpower', **box_args)
    ds_temp.set_density_contrast(smoothing_radius=R)
    ds_temp.set_quantiles(nquantiles=3, query_method='randoms')
    acf_temp = ds_temp.quantile_correlation(edges=edges, los='z', nthreads=4, gpu=True)
    
    # Plot monopole for middle quantile
    s, multipoles = acf_temp[1](ells=[0], return_sep=True)
    ax.plot(s, s**2 * multipoles[0], label=f'R = {R} Mpc/h', 
            color=color, linewidth=2, alpha=0.8)

ax.set_xlabel('s [Mpc/h]')
ax.set_ylabel(r'$s^2 \xi_0(s)$ [Mpc/h]$^2$')
ax.set_title('Smoothing Scale Comparison (Middle Quantile)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()